In [2]:
import pandas as pd
import openpyxl

In [3]:
# access data from repo
repo = "https://raw.githubusercontent.com/walde363/covid-housing-forecast/main/"
folder = "data/raw/"

urls = {
    "hurricanes": f"{repo}{folder}Huricanes.csv",
    "unemployment": f"{repo}{folder}Unemployment%20rate%20-%20FL.csv",
    "building_permits": f"{repo}{folder}New%20housing%20building%20permits%20-%20FL.csv",
    "mortgage_rates": f"{repo}{folder}historicalweeklydata_morgage_rates.xlsx",
    "hpi": f"{repo}{folder}hpi_po_monthly_hist.xlsx",
    "investor": f"{repo}{folder}dowload_investor_purchases_by_metro.csv",

    "metro_inventory_week": f"{repo}{folder}Metro_invt_fs_uc_sfr_week.csv",
    "metro_days_to_close_sm_week": f"{repo}{folder}Metro_mean_days_to_close_uc_sfrcondo_sm_week.csv",
    "metro_days_to_close_week": f"{repo}{folder}Metro_mean_days_to_close_uc_sfrcondo_week.csv",

    "metro_market_temp_month": f"{repo}{folder}Metro_market_temp_index_uc_sfrcondo_month.csv",
    "metro_new_con_sales_month": f"{repo}{folder}Metro_new_con_sales_count_raw_uc_sfrcondo_month.csv",
    "metro_zhvi_tier_month": f"{repo}{folder}Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_month.csv",
    "metro_zordi_condo_month": f"{repo}{folder}Metro_zordi_uc_condo_month.csv",
    "metro_zordi_mfr_month": f"{repo}{folder}Metro_zordi_uc_mfr_month.csv",
    "metro_zordi_sfr_month": f"{repo}{folder}Metro_zordi_uc_sfr_month.csv",
    "metro_zordi_all_month": f"{repo}{folder}Metro_zordi_uc_sfrcondomfr_month.csv"
}

def load_csv_with_fallback(url):
    # Handle mixed file exports: encoding issues and malformed rows.
    for encoding in ["utf-8", "utf-8-sig", "utf-16", "latin1"]:
        try:
            return pd.read_csv(url, encoding=encoding)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            try:
                return pd.read_csv(
                    url,
                    encoding=encoding,
                    sep=None,
                    engine="python",
                    on_bad_lines="skip"
                )
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue

    raise ValueError(f"Could not parse CSV with supported settings: {url}")

def load_data(url):
    if url.endswith(".csv"):
        return load_csv_with_fallback(url)
    elif url.endswith(".xlsx"):
        return pd.read_excel(url)
    else:
        raise ValueError(f"Unsupported file format: {url}")

data = {name: load_data(url) for name, url in urls.items()}

In [4]:
## Open https://www.aoml.noaa.gov/hrd/hurdat/All_U.S._Hurricanes.html 
## Copy and paste table to a google sheet
## Download sheet as a csv
hurricanes_df = data["hurricanes"]

## Open https://fred.stlouisfed.org/series/FLUR
## Press 10Y to limit to 10 years
## Press download
## Press csv
unemployment_df = data["unemployment"]

## Go to  https://fred.stlouisfed.org/series/FLBP1FH 
## Press 10Y to limit to 10 years
## Press download
## Press csv
fl_building_permits_df = data["building_permits"]

## Go to https://www.freddiemac.com/pmms?utm_source
## Press Download Rates Since 1971 XLSX
mortgage_rates_df = data["mortgage_rates"]

## Go to https://www.fhfa.gov/data/hpi/datasets
## Select U.S. and Census Division (Seasonally Adjusted and Not Adjusted) January 1991 - Latest Month    
hpi_df = data["hpi"]

## Go to https://www.redfin.com/news/data-center/investor-data/
## Directly under the words "Investor Purchases" click on Download Data
## Underneath the table click the box with an arrow pointing down
## Click cross tab
## Select CSV
## Press download
investor_df = data["investor"]

## Go to https://www.zillow.com/research/data/
## Navigate to dataset section on the page
## Select the data type and given geography
## Press download
metro_inventory_week_df        = data["metro_inventory_week"]
metro_days_to_close_sm_week_df = data["metro_days_to_close_sm_week"]
metro_market_temp_month_df     = data["metro_market_temp_month"]
metro_new_con_sales_month_df   = data["metro_new_con_sales_month"]
metro_zhvi_tier_month_df       = data["metro_zhvi_tier_month"]
metro_zordi_condo_month_df     = data["metro_zordi_condo_month"]
metro_zordi_mfr_month_df       = data["metro_zordi_mfr_month"]
metro_zordi_sfr_month_df       = data["metro_zordi_sfr_month"]
metro_zordi_all_month_df       = data["metro_zordi_all_month"]

In [5]:
def review_datasets(data_dict):
    for name, df in data_dict.items():
        print("="*60)
        print(f"Dataset: {name}")
        print("="*60)

        # Shape
        print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")

        # Data types
        print("Data Types:")
        print(df.dtypes)
        print()

        # Missing values
        missing = df.isnull().sum()
        missing_percent = (missing / len(df)) * 100

        missing_df = pd.DataFrame({
            "Missing Values": missing,
            "Percent (%)": missing_percent
        })

        print("Missing Values:")
        print(missing_df[missing_df["Missing Values"] > 0])
        print()

        # Duplicate rows
        duplicates = df.duplicated().sum()
        print(f"Duplicate Rows: {duplicates}\n")

        # Summary statistics (numeric columns only)
        print("Summary Statistics:")
        print(df.describe())
        print("\n\n")

In [6]:
review_datasets(data)

Dataset: hurricanes
Shape: 363 rows × 7 columns

Data Types:
ar                                           object
Month                                        object
States Affected and Category by States       object
Highest\nSaffir-\nSimpson\nU.S. Category     object
Central Pressure\n(mb)                      float64
Max Wind\n(kt)                               object
Name                                         object
dtype: object

Missing Values:
                                          Missing Values  Percent (%)
Month                                                 53    14.600551
States Affected and Category by States                53    14.600551
Highest\nSaffir-\nSimpson\nU.S. Category              53    14.600551
Central Pressure\n(mb)                                53    14.600551
Max Wind\n(kt)                                        53    14.600551
Name                                                  54    14.876033

Duplicate Rows: 1

Summary Statistics:
       Central

## Data processing

In [7]:
hurricanes_df.head()

,ar,Month,States Affected and Category by States,Highest\nSaffir-\nSimpson\nU.S. Category,Central Pressure\n(mb),Max Wind\n(kt),Name
0,1850s,NaN,NaN,NaN,NaN,NaN,NaN
1,1851,Jun,"TX, C1",1,974.0,80,-----
2,1851,Aug,"FL, NW3; I-GA, 1",3,955.0,100,"""Great Middle Florida"""
3,1852,Aug,"AL, 3; MS, 3; LA, 2; FL, SW2, NW1",3,961.0,100,"""Great Mobile"""
4,1852,Sep,"FL, SW1",1,982.0,70,-----


In [8]:
hurricanes_df['ar'].unique()

array(['1850s', '1851', '1852', '1853', '1854', '1855', '1856', '1857',
       '1858', '1859', '1860s', '1860', '1861', '1862', '1863', '1864',
       '1865', '1866', '1867', '1868', '1869', '1870s', '1870', '1871',
       '1872', '1873', '1874', '1875', '1876', '1877', '1878', '1879',
       '1880s', '1880', '1881', '1882', '1883', '1884', '1885', '1886',
       '1887', '1888', '1889', '1890s', '1890', '1891', '1892', '1893',
       '1894', '1895', '1896', '1897', '1898', '1899', '1900s', '1900',
       '1901', '1902', '1903', '1904', '1905', '1906', '1907', '1908',
       '1909', '1910s', '1910', '1911', '1912', '1913', '1914', '1915',
       '1916', '1917', '1918', '1919', '1920s', '1920', '1921', '1922',
       '1923', '1924', '1925', '1926', '1927', '1928', '1929', '1930s',
       '1930', '1931', '1932', '1933', '1934', '1935', '1936', '1937',
       '1938', '1939', '1940s', '1940', '1941', '1942', '1943', '1944',
       '1945', '1946', '1947', '1948', '1949', '1950s', '1950', '19

In [9]:
hurricanes_df.head(2)

,ar,Month,States Affected and Category by States,Highest\nSaffir-\nSimpson\nU.S. Category,Central Pressure\n(mb),Max Wind\n(kt),Name
0,1850s,NaN,NaN,NaN,NaN,NaN,NaN
1,1851,Jun,"TX, C1",1,974.0,80,-----


In [10]:
# remove rows that contain s. This is a row to separate years
hurricanes_df_clean = hurricanes_df.copy(deep=True)
hurricanes_df_clean = hurricanes_df_clean[~hurricanes_df_clean["ar"].str.contains(r"[A-Za-z]", na=False)]

# flag rows with FL
hurricanes_df_clean["fl_flag"] = hurricanes_df_clean["States Affected and Category by States"].str.contains("FL", na=False)

hurricanes_df_clean = hurricanes_df_clean.rename(columns={
    'ar': 'year',
    'Highest\nSaffir-\nSimpson\nU.S. Category': 'highest_category'
})

#filter to FL
hurricanes_df_clean = hurricanes_df_clean[hurricanes_df_clean['fl_flag'] == True]

In [11]:
month_map = {
    "Jan": 1, "Feb": 2, "Mar": 3,
    "Apr": 4, "May": 5, "Jun": 6,
    "Jul": 7, "Aug": 8, "Sep": 9,
    "Oct": 10, "Nov": 11, "Dec": 12
}

# Make this cell safe to re-run: only expand when the raw Month column exists.
if "Month" in hurricanes_df_clean.columns:
    def expand_month(row):
        month_value = row["Month"]

        if month_value == "Sp-Oc":
            months = [9, 10]
        elif month_value == "Jl-Au":
            months = [7, 8]
        else:
            months = [month_map[month_value]]

        return pd.DataFrame({
            "year": row["year"],
            "month": months,
            "highest_category": pd.to_numeric(row["highest_category"], errors="coerce")
        })

    hurricanes_df_clean = pd.concat(
        hurricanes_df_clean.apply(expand_month, axis=1).tolist(),
        ignore_index=True
    )
elif {"year", "month", "highest_category"}.issubset(hurricanes_df_clean.columns):
    hurricanes_df_clean = hurricanes_df_clean.copy()
else:
    raise KeyError(
        "Expected either raw hurricane columns including 'Month' or transformed columns: "
        "'year', 'month', 'highest_category'."
    )

In [12]:
hurricanes_df_clean.head(100)

,year,month,highest_category
0,1851,8,3.0
1,1852,8,3.0
2,1852,9,1.0
3,1852,10,2.0
4,1854,9,3.0
...,...,...,...
95,1964,10,2.0
96,1965,9,4.0
97,1966,6,3.0
98,1966,10,2.0


In [13]:
unemployment_df

,observation_date,FLUR
0,2015-12-01,5.1
1,2016-01-01,5.1
2,2016-02-01,5.0
3,2016-03-01,4.9
4,2016-04-01,4.9
...,...,...
116,2025-08-01,3.8
117,2025-09-01,3.9
118,2025-10-01,NaN
119,2025-11-01,4.2


In [14]:
unemployment_df['FLUR'].unique()

array([ 5.1,  5. ,  4.9,  4.8,  4.7,  4.6,  4.5,  4.4,  4.3,  4.2,  4.1,
        4. ,  3.9,  3.8,  3.7,  3.6,  3.5,  3.4,  3.3,  3.2,  3.1,  3. ,
       13.3, 14.1, 11.6,  8.6,  8. ,  6.9,  6.5,  6.2,  5.8,  5.5,  5.4,
        5.2,  2.9,  2.8,  nan])

In [15]:
unemployment_df['observation_date'].unique()

array(['2015-12-01', '2016-01-01', '2016-02-01', '2016-03-01',
       '2016-04-01', '2016-05-01', '2016-06-01', '2016-07-01',
       '2016-08-01', '2016-09-01', '2016-10-01', '2016-11-01',
       '2016-12-01', '2017-01-01', '2017-02-01', '2017-03-01',
       '2017-04-01', '2017-05-01', '2017-06-01', '2017-07-01',
       '2017-08-01', '2017-09-01', '2017-10-01', '2017-11-01',
       '2017-12-01', '2018-01-01', '2018-02-01', '2018-03-01',
       '2018-04-01', '2018-05-01', '2018-06-01', '2018-07-01',
       '2018-08-01', '2018-09-01', '2018-10-01', '2018-11-01',
       '2018-12-01', '2019-01-01', '2019-02-01', '2019-03-01',
       '2019-04-01', '2019-05-01', '2019-06-01', '2019-07-01',
       '2019-08-01', '2019-09-01', '2019-10-01', '2019-11-01',
       '2019-12-01', '2020-01-01', '2020-02-01', '2020-03-01',
       '2020-04-01', '2020-05-01', '2020-06-01', '2020-07-01',
       '2020-08-01', '2020-09-01', '2020-10-01', '2020-11-01',
       '2020-12-01', '2021-01-01', '2021-02-01', '2021-

In [16]:
unemployment_df_clean = unemployment_df.copy(deep=True)
unemployment_df_clean = unemployment_df_clean[unemployment_df_clean['FLUR'].notnull()]

In [17]:
df = unemployment_df_clean
missing = df.isnull().sum()
(missing / len(df)) * 100

observation_date    0.0
FLUR                0.0
dtype: float64

In [18]:
fl_building_permits_df_clean = fl_building_permits_df.copy(deep=True)
fl_building_permits_df_clean["observation_date"] = pd.to_datetime(fl_building_permits_df_clean["observation_date"])
fl_building_permits_df_clean["year"] = fl_building_permits_df_clean["observation_date"].dt.year
fl_building_permits_df_clean["month"] = fl_building_permits_df_clean["observation_date"].dt.month

In [19]:
mortgage_rates_df.head(10)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,NaN,NaN,NaN,PRIMARY MORTGAGE MARKET SURVEY®,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,Summary page with all rate types - U.S. averages,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,U.S.,30 yr,U.S.,15 yr,U.S.,5/1 ARM,U.S.,30 yr FRM/
4,NaN,30 yr,fees &,15 yr,fees &,5/1,fees &,5/1 ARM,5/1 ARM
5,Week,FRM,points,FRM,points,ARM,points,margin,spread
6,1971-04-02 00:00:00,7.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1971-04-09 00:00:00,7.31,,NaN,NaN,NaN,NaN,NaN,NaN
8,1971-04-16 00:00:00,7.31,,NaN,NaN,NaN,NaN,NaN,NaN
9,1971-04-23 00:00:00,7.31,,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
mortgage_rates_df_clean = mortgage_rates_df.copy(deep=True)
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Unnamed: 0'].notnull()]

# take first row as header
mortgage_rates_df_clean.columns = mortgage_rates_df_clean.iloc[0]
mortgage_rates_df_clean = mortgage_rates_df_clean[1:].reset_index(drop=True)

week_parsed = pd.to_datetime(mortgage_rates_df_clean["Week"], errors="coerce")
mortgage_rates_df_clean = mortgage_rates_df_clean[week_parsed.notna()].copy()
mortgage_rates_df_clean["Week"] = week_parsed[week_parsed.notna()].dt.date

# renaming columns from base spreedsheet
mortgage_rates_df_clean.columns.values[0] = "Week"
mortgage_rates_df_clean.columns.values[1] = "U.S. 30 year FRM"
mortgage_rates_df_clean.columns.values[2] = "30 year fees & points"

mortgage_rates_df_clean.columns.values[3] = "U.S. 15 year FRM"
mortgage_rates_df_clean.columns.values[4] = "15 year fees & points"

mortgage_rates_df_clean.columns.values[5] = "U.S. 5/1 ARM"
mortgage_rates_df_clean.columns.values[6] = "5/1 year fees & points"

mortgage_rates_df_clean.columns.values[7] = "U.S. 5/1 ARM margin"
mortgage_rates_df_clean.columns.values[8] = "30 year FRM / 5/1 ARM spread"

#Truncate
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Week']>=pd.to_datetime("2016-01-01").date()]

unemployment_df_clean["observation_date"] = pd.to_datetime(unemployment_df_clean["observation_date"])
unemployment_df_clean["year"] = unemployment_df_clean["observation_date"].dt.year
unemployment_df_clean["month"] = unemployment_df_clean["observation_date"].dt.month

In [21]:
hpi_df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,Monthly FHFA House Price Index® for Census Div...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Month,East North Central\n(NSA),East North Central\n(SA),East South Central\n(NSA),East South Central\n(SA),Middle Atlantic\n(NSA),Middle Atlantic\n(SA),Mountain\n\n(NSA),Mountain\n\n(SA),New England\n(NSA),...,Pacific\n\n(NSA),Pacific\n\n(SA),South Atlantic\n(NSA),South Atlantic\n(SA),West North Central\n(NSA),West North Central\n(SA),West South Central\n(NSA),West South Central\n(SA),USA\n\n(NSA),USA\n\n(SA)
3,1991-01-01 00:00:00,100,100,100,100,100,100,100,100,100,...,100,100,100,100,100,100,100,100,100,100
4,1991-02-01 00:00:00,100.87,100.87,100.91,100.5,100.05,100.13,98.38,98.82,101.75,...,100.18,100.65,100.55,100.32,100.44,100.4,99.81,99.68,100.36,100.4


In [22]:
hpi_df_clean = hpi_df.copy(deep=True)


hpi_df_clean.columns = hpi_df_clean.iloc[2]
# Drop the first three rows
hpi_df_clean = hpi_df_clean.iloc[3:].reset_index(drop=True)


week_parsed = pd.to_datetime(hpi_df_clean["Month"], errors="coerce")
hpi_df_clean = hpi_df_clean[week_parsed.notna()].copy()
hpi_df_clean["Month"] = week_parsed[week_parsed.notna()].dt.date
hpi_df_clean.columns = hpi_df_clean.columns.str.replace('\n', ' ')
hpi_df_clean = pd.melt(hpi_df_clean, id_vars="Month", var_name="Location", value_name="HPI")

hpi_df_clean["observation_date"] = pd.to_datetime(hpi_df_clean["Month"])
hpi_df_clean["year"] = hpi_df_clean["observation_date"].dt.year
hpi_df_clean["month"] = hpi_df_clean["observation_date"].dt.month

In [23]:
hpi_df_clean

,Month,Location,HPI,observation_date,year,month
0,1991-01-01,East North Central (NSA),100,1991-01-01,1991,1
1,1991-02-01,East North Central (NSA),100.87,1991-02-01,1991,2
2,1991-03-01,East North Central (NSA),101.32,1991-03-01,1991,3
3,1991-04-01,East North Central (NSA),101.73,1991-04-01,1991,4
4,1991-05-01,East North Central (NSA),102.32,1991-05-01,1991,5
...,...,...,...,...,...,...
8395,2025-08-01,USA (SA),435.3,2025-08-01,2025,8
8396,2025-09-01,USA (SA),435.04,2025-09-01,2025,9
8397,2025-10-01,USA (SA),436.78,2025-10-01,2025,10
8398,2025-11-01,USA (SA),439.72,2025-11-01,2025,11


In [24]:
investor_df.head(3)

,Quarter,Redfin Metro,Investor Market Share,Investor Purchases,Investor Purchases YoY
0,2025 Q4,National,18%,"49,824",+2%
1,2025 Q4,"Anaheim, CA",26%,"1,260",+3%
2,2025 Q4,"Atlanta, GA",20%,"3,277",+7%


In [54]:
investor_df_clean = investor_df.copy(deep=True)

# Split year and quarter
investor_df_clean[["year", "q"]] = investor_df_clean["Quarter"].str.split(" ", expand=True)

investor_df_clean["year"] = investor_df_clean["year"].astype(int)
investor_df_clean["quarter"] = investor_df_clean["q"].str.replace("Q", "").astype(int)

# Convert quarter → starting month
investor_df_clean["month"] = (investor_df_clean["quarter"] - 1) * 3 + 1

# Create datetime (quarter start)
investor_df_clean["date"] = pd.to_datetime(
    dict(year=investor_df_clean["year"], month=investor_df_clean["month"], day=1)
)

investor_df_clean["Investor Purchases"] = (
    investor_df_clean["Investor Purchases"]
    .str.replace(",", "", regex=False)
    .astype("Int32")
)

investor_df_clean["Investor Market Share"] = (
    investor_df_clean["Investor Market Share"]
    .str.replace("%", "", regex=False)
    .astype("Int32")
)

investor_df_clean["Investor Purchases YoY"] = (
    investor_df_clean["Investor Purchases YoY"]
    .str.replace("%", "", regex=False)
    .astype("Int32")
)

investor_df_clean = investor_df_clean.groupby(["year", "month", "date", "Redfin Metro"]).agg(
    {"Investor Purchases": "sum"
    ,"Investor Market Share": "mean"
    ,"Investor Purchases YoY": "mean"}).reset_index()

investor_df_clean[['city', 'state']] = investor_df_clean['Redfin Metro'].str.split(',', expand=True)
investor_df_clean = investor_df_clean.groupby(["date","state"]).agg({"Investor Purchases": "sum", "Investor Market Share": "mean", "Investor Purchases YoY": "mean"}).reset_index()

In [26]:
metro_inventory_week_df.head(1)

,RegionID,SizeRank,RegionName,RegionType,StateName,2018-01-13,2018-01-20,2018-01-27,2018-02-03,2018-02-10,...,2025-12-06,2025-12-13,2025-12-20,2025-12-27,2026-01-03,2026-01-10,2026-01-17,2026-01-24,2026-01-31,2026-02-07
0,102001,0,United States,country,NaN,974217.0,972070.0,973707.0,981261.0,979730.0,...,842036.0,822863.0,800717.0,762491.0,756562.0,753855.0,759108.0,753685.0,742157.0,734694.0


In [27]:
metro_inventory_week_df_clean = metro_inventory_week_df.copy(deep=True)
metro_inventory_week_df_clean = pd.melt(
    metro_inventory_week_df_clean,
    id_vars=["RegionID", "SizeRank", "RegionName", "RegionType","StateName"],
    var_name="Week",
    value_name="Inventory"
)

metro_inventory_week_df_clean = metro_inventory_week_df_clean[metro_inventory_week_df_clean["Inventory"].notnull()]

metro_inventory_week_df_clean["Week"] = pd.to_datetime(metro_inventory_week_df_clean["Week"])
metro_inventory_week_df_clean["year"] = metro_inventory_week_df_clean["Week"].dt.year
metro_inventory_week_df_clean["month"] = metro_inventory_week_df_clean["Week"].dt.month

metro_inventory_week_df_clean = metro_inventory_week_df_clean[metro_inventory_week_df_clean["Inventory"].notnull()]

metro_inventory_week_df_clean = metro_inventory_week_df_clean.groupby(["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "year", "month"])["Inventory"].mean().reset_index()

In [28]:
metro_days_to_close_sm_week_df_clean = metro_days_to_close_sm_week_df.copy(deep=True)
metro_days_to_close_sm_week_df_clean = pd.melt(
    metro_days_to_close_sm_week_df_clean,
    id_vars=["RegionID", "SizeRank", "RegionName", "RegionType","StateName"],
    var_name="Week",
    value_name="DaysToClose"
)

metro_days_to_close_sm_week_df_clean = metro_days_to_close_sm_week_df_clean[metro_days_to_close_sm_week_df_clean["DaysToClose"].notnull()]

metro_days_to_close_sm_week_df_clean["Week"] = pd.to_datetime(metro_days_to_close_sm_week_df_clean["Week"])
metro_days_to_close_sm_week_df_clean["year"] = metro_days_to_close_sm_week_df_clean["Week"].dt.year
metro_days_to_close_sm_week_df_clean["month"] = metro_days_to_close_sm_week_df_clean["Week"].dt.month

metro_days_to_close_sm_week_df_clean = metro_days_to_close_sm_week_df_clean[metro_days_to_close_sm_week_df_clean["DaysToClose"].notnull()]

metro_days_to_close_sm_week_df_clean = metro_days_to_close_sm_week_df_clean.groupby(["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "year", "month"])["DaysToClose"].mean().reset_index()

In [29]:
metro_market_temp_month_df_clean = metro_market_temp_month_df.copy(deep=True)
metro_market_temp_month_df_clean = pd.melt(
    metro_market_temp_month_df_clean,
    id_vars=["RegionID", "SizeRank", "RegionName", "RegionType","StateName"],
    var_name="Month",
    value_name="MarketTemp"
)

metro_market_temp_month_df_clean = metro_market_temp_month_df_clean[metro_market_temp_month_df_clean["MarketTemp"].notnull()]

metro_market_temp_month_df_clean["Month"] = pd.to_datetime(metro_market_temp_month_df_clean["Month"])
metro_market_temp_month_df_clean["year"] = metro_market_temp_month_df_clean["Month"].dt.year
metro_market_temp_month_df_clean["month"] = metro_market_temp_month_df_clean["Month"].dt.month

In [30]:
metro_new_con_sales_month_df_clean = metro_new_con_sales_month_df.copy(deep=True)
metro_new_con_sales_month_df_clean = pd.melt(
    metro_new_con_sales_month_df_clean,
    id_vars=["RegionID", "SizeRank", "RegionName", "RegionType","StateName"],
    var_name="Month",
    value_name="NewConSales"
)

metro_new_con_sales_month_df_clean = metro_new_con_sales_month_df_clean[metro_new_con_sales_month_df_clean["NewConSales"].notnull()]

metro_new_con_sales_month_df_clean["Month"] = pd.to_datetime(metro_new_con_sales_month_df_clean["Month"])
metro_new_con_sales_month_df_clean["year"] = metro_new_con_sales_month_df_clean["Month"].dt.year
metro_new_con_sales_month_df_clean["month"] = metro_new_con_sales_month_df_clean["Month"].dt.month

In [31]:
metro_zhvi_tier_month_df_clean = metro_zhvi_tier_month_df.copy(deep=True)
metro_zhvi_tier_month_df_clean = pd.melt(
    metro_zhvi_tier_month_df_clean,
    id_vars=["RegionID", "SizeRank", "RegionName", "RegionType","StateName"],
    var_name="Month",
    value_name="ZHVI_Tier"
)

metro_zhvi_tier_month_df_clean = metro_zhvi_tier_month_df_clean[metro_zhvi_tier_month_df_clean["ZHVI_Tier"].notnull()]

metro_zhvi_tier_month_df_clean["Month"] = pd.to_datetime(metro_zhvi_tier_month_df_clean["Month"])
metro_zhvi_tier_month_df_clean["year"] = metro_zhvi_tier_month_df_clean["Month"].dt.year
metro_zhvi_tier_month_df_clean["month"] = metro_zhvi_tier_month_df_clean["Month"].dt.month

In [32]:
metro_zordi_condo_month_df_clean = metro_zordi_condo_month_df.copy(deep=True)
metro_zordi_condo_month_df_clean = pd.melt(
    metro_zordi_condo_month_df_clean,
    id_vars=["RegionID", "SizeRank", "RegionName", "RegionType","StateName"],
    var_name="Month",
    value_name="ZORDI_Condo"
)

metro_zordi_condo_month_df_clean = metro_zordi_condo_month_df_clean[metro_zordi_condo_month_df_clean["ZORDI_Condo"].notnull()]

metro_zordi_condo_month_df_clean["Month"] = pd.to_datetime(metro_zordi_condo_month_df_clean["Month"])
metro_zordi_condo_month_df_clean["year"] = metro_zordi_condo_month_df_clean["Month"].dt.year
metro_zordi_condo_month_df_clean["month"] = metro_zordi_condo_month_df_clean["Month"].dt.month

In [33]:
metro_zordi_mfr_month_df_clean = metro_zordi_mfr_month_df.copy(deep=True)
metro_zordi_mfr_month_df_clean = pd.melt(
    metro_zordi_mfr_month_df_clean,
    id_vars=["RegionID", "SizeRank", "RegionName", "RegionType","StateName"],
    var_name="Month",
    value_name="ZORDI_MFR"
)

metro_zordi_mfr_month_df_clean = metro_zordi_mfr_month_df_clean[metro_zordi_mfr_month_df_clean["ZORDI_MFR"].notnull()]

metro_zordi_mfr_month_df_clean["Month"] = pd.to_datetime(metro_zordi_mfr_month_df_clean["Month"])
metro_zordi_mfr_month_df_clean["year"] = metro_zordi_mfr_month_df_clean["Month"].dt.year
metro_zordi_mfr_month_df_clean["month"] = metro_zordi_mfr_month_df_clean["Month"].dt.month

In [34]:
metro_zordi_sfr_month_df_clean = metro_zordi_sfr_month_df.copy(deep=True)
metro_zordi_sfr_month_df_clean = pd.melt(
    metro_zordi_sfr_month_df_clean,
    id_vars=["RegionID", "SizeRank", "RegionName", "RegionType","StateName"],
    var_name="Month",
    value_name="ZORDI_SFR"
)

metro_zordi_sfr_month_df_clean = metro_zordi_sfr_month_df_clean[metro_zordi_sfr_month_df_clean["ZORDI_SFR"].notnull()]


metro_zordi_sfr_month_df_clean["Month"] = pd.to_datetime(metro_zordi_sfr_month_df_clean["Month"])
metro_zordi_sfr_month_df_clean["year"] = metro_zordi_sfr_month_df_clean["Month"].dt.year
metro_zordi_sfr_month_df_clean["month"] = metro_zordi_sfr_month_df_clean["Month"].dt.month

In [35]:
metro_zordi_all_month_df_clean = metro_zordi_all_month_df.copy(deep=True)
metro_zordi_all_month_df_clean = pd.melt(
    metro_zordi_all_month_df_clean,
    id_vars=["RegionID", "SizeRank", "RegionName", "RegionType","StateName"],
    var_name="Month",
    value_name="ZORDI_All"
)

metro_zordi_all_month_df_clean = metro_zordi_all_month_df_clean[metro_zordi_all_month_df_clean["ZORDI_All"].notnull()]

metro_zordi_all_month_df_clean["Month"] = pd.to_datetime(metro_zordi_all_month_df_clean["Month"])
metro_zordi_all_month_df_clean["year"] = metro_zordi_all_month_df_clean["Month"].dt.year
metro_zordi_all_month_df_clean["month"] = metro_zordi_all_month_df_clean["Month"].dt.month

In [36]:
hurricanes_df_clean.isna().sum()

year                0
month               0
highest_category    1
dtype: int64

In [37]:
unemployment_df_clean.isna().sum()

observation_date    0
FLUR                0
year                0
month               0
dtype: int64

In [38]:
print(hurricanes_df_clean.isna().sum(), "\n")
print(unemployment_df_clean.isna().sum(), "\n")
print(fl_building_permits_df_clean.isna().sum(), "\n")
print(mortgage_rates_df_clean.isna().sum(), "\n")
print(hpi_df_clean.isna().sum(), "\n")
print(investor_df_clean.isna().sum(), "\n")
print(metro_inventory_week_df_clean.isna().sum(), "\n")
print(metro_days_to_close_sm_week_df_clean.isna().sum(), "\n")
print(metro_market_temp_month_df_clean.isna().sum(), "\n")
print(metro_new_con_sales_month_df_clean.isna().sum(), "\n")
print(metro_zhvi_tier_month_df_clean.isna().sum(), "\n")
print(metro_zordi_condo_month_df_clean.isna().sum(), "\n")
print(metro_zordi_mfr_month_df_clean.isna().sum(), "\n")
print(metro_zordi_sfr_month_df_clean.isna().sum(), "\n")
print(metro_zordi_all_month_df_clean.isna().sum(), "\n")


year                0
month               0
highest_category    1
dtype: int64 

observation_date    0
FLUR                0
year                0
month               0
dtype: int64 

observation_date    0
FLBP1FH             0
year                0
month               0
dtype: int64 

5
Week                              0
U.S. 30 year FRM                  0
30 year fees & points           173
U.S. 15 year FRM                  0
15 year fees & points           173
U.S. 5/1 ARM                    173
5/1 year fees & points          173
U.S. 5/1 ARM margin             173
30 year FRM / 5/1 ARM spread    173
dtype: int64 

Month               0
Location            0
HPI                 0
observation_date    0
year                0
month               0
dtype: int64 

year                        0
month                       0
date                        0
Redfin Metro                0
Investor Purchases          0
Investor Market Share       0
Investor Purchases YoY    156
dtype: int64 



In [39]:
hurricanes_df_clean["date"] = pd.to_datetime(
    dict(year=hurricanes_df_clean["year"], month=hurricanes_df_clean["month"], day=1)
)

unemployment_df_clean["date"] = pd.to_datetime(
    dict(year=unemployment_df_clean["year"], month=unemployment_df_clean["month"], day=1)
)

fl_building_permits_df_clean["date"] = pd.to_datetime(
    dict(year=fl_building_permits_df_clean["year"], month=fl_building_permits_df_clean["month"], day=1)
)

hpi_df_clean["date"] = pd.to_datetime(
    dict(year=hpi_df_clean["year"], month=hpi_df_clean["month"], day=1)
)

investor_df_clean["date"] = pd.to_datetime(
    dict(year=investor_df_clean["year"], month=investor_df_clean["month"], day=1)
)

metro_inventory_week_df_clean["date"] = pd.to_datetime(
    dict(year=metro_inventory_week_df_clean["year"], month=metro_inventory_week_df_clean["month"], day=1)
)

metro_days_to_close_sm_week_df_clean["date"] = pd.to_datetime(
    dict(year=metro_days_to_close_sm_week_df_clean["year"], month=metro_days_to_close_sm_week_df_clean["month"], day=1)
)

metro_market_temp_month_df_clean["date"] = pd.to_datetime(
    dict(year=metro_market_temp_month_df_clean["year"], month=metro_market_temp_month_df_clean["month"], day=1)
)

metro_new_con_sales_month_df_clean["date"] = pd.to_datetime(
    dict(year=metro_new_con_sales_month_df_clean["year"], month=metro_new_con_sales_month_df_clean["month"], day=1)
)

metro_zhvi_tier_month_df_clean["date"] = pd.to_datetime(
    dict(year=metro_zhvi_tier_month_df_clean["year"], month=metro_zhvi_tier_month_df_clean["month"], day=1)
)

metro_zordi_condo_month_df_clean["date"] = pd.to_datetime(
    dict(year=metro_zordi_condo_month_df_clean["year"], month=metro_zordi_condo_month_df_clean["month"], day=1)
)   

metro_zordi_mfr_month_df_clean["date"] = pd.to_datetime(
    dict(year=metro_zordi_mfr_month_df_clean["year"], month=metro_zordi_mfr_month_df_clean["month"], day=1)
)

metro_zordi_sfr_month_df_clean["date"] = pd.to_datetime(
    dict(year=metro_zordi_sfr_month_df_clean["year"], month=metro_zordi_sfr_month_df_clean["month"], day=1)
)

metro_zordi_all_month_df_clean["date"] = pd.to_datetime(
    dict(year=metro_zordi_all_month_df_clean["year"], month=metro_zordi_all_month_df_clean["month"], day=1)
)


In [40]:
dfs = [
hurricanes_df_clean,
unemployment_df_clean,
fl_building_permits_df_clean,
hpi_df_clean,
investor_df_clean,
metro_inventory_week_df_clean,
metro_days_to_close_sm_week_df_clean,
metro_market_temp_month_df_clean,
metro_new_con_sales_month_df_clean,
metro_zhvi_tier_month_df_clean,
metro_zordi_condo_month_df_clean,
metro_zordi_mfr_month_df_clean,
metro_zordi_sfr_month_df_clean,
metro_zordi_all_month_df_clean
]


max_min_date = max(df["date"].min() for df in dfs)

min_max_date = min(df["date"].max() for df in dfs)

print("Maximum of Minimum Dates:", max_min_date)
print("Minimum of Maximum Dates:", min_max_date)


Maximum of Minimum Dates: 2020-06-01 00:00:00
Minimum of Maximum Dates: 2024-10-01 00:00:00


In [41]:
hurricanes_df_clean = hurricanes_df_clean[hurricanes_df_clean["date"] >= max_min_date]
unemployment_df_clean = unemployment_df_clean[unemployment_df_clean["date"] >= max_min_date]
fl_building_permits_df_clean = fl_building_permits_df_clean[fl_building_permits_df_clean["date"] >= max_min_date]
hpi_df_clean = hpi_df_clean[hpi_df_clean["date"] >= max_min_date]
investor_df_clean = investor_df_clean[investor_df_clean["date"] >= max_min_date]
metro_inventory_week_df_clean = metro_inventory_week_df_clean[metro_inventory_week_df_clean["date"] >= max_min_date]
metro_days_to_close_sm_week_df_clean = metro_days_to_close_sm_week_df_clean[metro_days_to_close_sm_week_df_clean["date"] >= max_min_date]
metro_market_temp_month_df_clean = metro_market_temp_month_df_clean[metro_market_temp_month_df_clean["date"] >= max_min_date]
metro_new_con_sales_month_df_clean = metro_new_con_sales_month_df_clean[metro_new_con_sales_month_df_clean["date"] >= max_min_date]
metro_zhvi_tier_month_df_clean = metro_zhvi_tier_month_df_clean[metro_zhvi_tier_month_df_clean["date"] >= max_min_date]
metro_zordi_condo_month_df_clean = metro_zordi_condo_month_df_clean[metro_zordi_condo_month_df_clean["date"] >= max_min_date]
metro_zordi_mfr_month_df_clean = metro_zordi_mfr_month_df_clean[metro_zordi_mfr_month_df_clean["date"] >= max_min_date]
metro_zordi_sfr_month_df_clean = metro_zordi_sfr_month_df_clean[metro_zordi_sfr_month_df_clean["date"] >= max_min_date]
metro_zordi_all_month_df_clean = metro_zordi_all_month_df_clean[metro_zordi_all_month_df_clean["date"] >= max_min_date]

In [42]:
hurricanes_df_clean.drop(columns=["year", "month"], inplace=True)
unemployment_df_clean.drop(columns=["year", "month"], inplace=True)
fl_building_permits_df_clean.drop(columns=["year", "month"], inplace=True)
hpi_df_clean.drop(columns=["year", "month"], inplace=True)
investor_df_clean.drop(columns=["year", "month"], inplace=True)
metro_inventory_week_df_clean.drop(columns=["year", "month"], inplace=True)
metro_days_to_close_sm_week_df_clean.drop(columns=["year", "month"], inplace=True)
metro_market_temp_month_df_clean.drop(columns=["year", "month"], inplace=True)
metro_new_con_sales_month_df_clean.drop(columns=["year", "month"], inplace=True)
metro_zhvi_tier_month_df_clean.drop(columns=["year", "month"], inplace=True)
metro_zordi_condo_month_df_clean.drop(columns=["year", "month"], inplace=True)
metro_zordi_mfr_month_df_clean.drop(columns=["year", "month"], inplace=True)
metro_zordi_sfr_month_df_clean.drop(columns=["year", "month"], inplace=True)
metro_zordi_all_month_df_clean.drop(columns=["year", "month"], inplace=True)

/tmp/ipykernel_2672/3686387145.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hurricanes_df_clean.drop(columns=["year", "month"], inplace=True)
/tmp/ipykernel_2672/3686387145.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unemployment_df_clean.drop(columns=["year", "month"], inplace=True)
/tmp/ipykernel_2672/3686387145.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fl_building_permits_df_clean.drop(columns=["year", "mont

In [89]:
investor_df_clean.head(1000)

,date,state,Investor Purchases,Investor Market Share,Investor Purchases YoY
0,2000-01-01,AZ,1811,11.0,<NA>
1,2000-01-01,CA,4476,5.375,<NA>
2,2000-01-01,CO,632,6.0,<NA>
3,2000-01-01,DC,548,5.0,<NA>
4,2000-01-01,FL,4475,9.166667,<NA>
...,...,...,...,...,...
995,2011-04-01,MI,212,5.0,2.0
996,2011-04-01,MN,494,8.0,-21.0
997,2011-04-01,NC,301,9.0,-33.0
998,2011-04-01,NJ,516,7.0,7.0


In [91]:
investor_df_clean["state"] = investor_df_clean["state"].astype(str)
investor_df_clean.dtypes

date                      datetime64[ns]
state                             object
Investor Purchases                 Int32
Investor Market Share            Float64
Investor Purchases YoY           Float64
dtype: object

In [ ]:
investor_df_clean_wide = investor_df_clean.copy(deep=True)

investor_df_clean_wide_FL = investor_df_clean_wide[investor_df_clean_wide["state"].str.strip() == "FL"].copy()
investor_df_clean_wide_Total = investor_df_clean_wide.copy()
investor_df_clean_wide_Total = investor_df_clean_wide_Total.groupby("date").agg({
    "Investor Purchases": "sum",
    "Investor Market Share": "mean",
    "Investor Purchases YoY": "mean"
}).reset_index()

investor_df_clean_wide = pd.merge(investor_df_clean_wide_FL,investor_df_clean_wide_Total, how="outer", on="date", suffixes=("_FL", "_Total"))
investor_df_clean_wide = investor_df_clean_wide.drop(columns=["state"])

,date,Investor Purchases_FL,Investor Market Share_FL,Investor Purchases YoY_FL,Investor Purchases_Total,Investor Market Share_Total,Investor Purchases YoY_Total
0,2000-01-01,4475,9.166667,<NA>,25402,7.565476,<NA>
1,2000-04-01,5019,8.0,<NA>,29717,6.52381,<NA>
2,2000-07-01,4926,8.166667,<NA>,27433,6.107143,<NA>
3,2000-10-01,4825,8.5,<NA>,25745,6.583333,<NA>
4,2001-01-01,4637,9.5,4.5,24486,7.445076,-4.083333
...,...,...,...,...,...,...,...
99,2024-10-01,9188,22.833333,-14.833333,47872,15.589015,-0.005682
100,2025-01-01,9146,23.333333,-10.166667,47009,17.231061,4.113636
101,2025-04-01,9649,20.333333,-17.333333,52128,14.825758,-2.24053
102,2025-07-01,8896,20.166667,-10.5,51707,14.875,5.727273


In [108]:
hpi_df_clean = hpi_df_clean.copy(deep=True)

hpi_df_clean['Location'].unique()

array(['East North Central (NSA)', 'East North Central (SA)',
       'East South Central (NSA)', 'East South Central (SA)',
       'Middle Atlantic (NSA)', 'Middle Atlantic (SA)', 'Mountain  (NSA)',
       'Mountain  (SA)', 'New England (NSA)', 'New England (SA)',
       'Pacific  (NSA)', 'Pacific  (SA)', 'South Atlantic (NSA)',
       'South Atlantic (SA)', 'West North Central (NSA)',
       'West North Central (SA)', 'West South Central (NSA)',
       'West South Central (SA)', 'USA  (NSA)', 'USA  (SA)'], dtype=object)

In [106]:
processed_data = pd.merge(hurricanes_df_clean, unemployment_df_clean, on="date", how="outer")
print("After merging hurricanes and unemployment:", processed_data.shape)
processed_data = pd.merge(processed_data, fl_building_permits_df_clean, on="date", how="outer")
print("After merging building permits:", processed_data.shape)
processed_data = pd.merge(processed_data, investor_df_clean_wide, on="date", how="outer")
print("After merging investor data:", processed_data.shape)
processed_data = pd.merge(processed_data, hpi_df_clean, on="date", how="outer")
print("After merging HPI:", processed_data.shape)

After merging hurricanes and unemployment: (66, 4)
After merging building permits: (67, 6)
After merging investor data: (149, 12)
After merging HPI: (1422, 16)
